# Подготовка формата данных

В файле ```landmarks_data_wiki.json``` собраны данные о достопримечательностях. Формат хранения данных следующий:
* ```wikidata_id``` - код объекта на wikidata
* ```name_en``` - название достопримечательности на английском
* ```name_ru``` - название достопримечательности на русском
* ```name_de``` - название достопримечательности на немецком
* ```wikidata_description_en``` - краткое описание на английском из wikidata
* ```wikidata_description_ru``` - краткое описание на русском из wikidata
* ```coordinates```:
  - ```latitude``` - широта
  - ```longitude``` - долгота
* ```city_ru``` - название региона, города или района на русском
* ```city_en``` - название региона, города или района на английском
* ```year_built``` - год постройки
* ```country_en``` - название страны на английском
* ```country_ru``` - название страны на русском
* ```architect_ru``` - имя архитектора на русском
* ```architect_en``` - имя архитектора на английском
* ```architectural_style_ru``` - архитектурный стиль на русском
* ```architectural_style_en``` - архитектурный стиль на английском
* ```heritage_status_ru``` - статус достопримечательности на русском
* ```heritage_status_en``` - статус достопримечательности на английском
* ```website``` - ссфлка на сайт достопримечательности
* ```wikipedia_url_en``` - ссылка на статью в wikipedia на английском
* ```wikipedia_url_ru``` - ссылка на статью в wikipedia на русском
* ```wikipedia_url_de``` - ссылка на статью в wikipedia на немецком
* ```landmark_type```:
  -  ```ru``` - тип достопримечательности на русском с Wikidata
  - ```en``` - тип достопримечательности на английском с Wikidata
  - ```type_id``` - тип достопримечательности на русском
* ```wikipedia_summary_en``` - описание достопримечательности с wikipedia на русском
* ```wikipedia_summary_ru``` - описание достопримечательности с wikipedia на английском
* ```wikipedia_summary_de``` - описание достопримечательности с wikipedia на немецком
* ```url``` - ссылка на достопримечательность на Wikidata
* ```landmark_id``` - id соответсвующее id достопримечательности из датасета Google Landmarks V2
* ```hierarchical_label``` - тип достопримечательности из датасета Google Landmarks V2


Необходимо скачать изображения, соответствующие выбранным достопримечательностям, и добавить в данные поле ```image_path```, ```ground_truth``` будет соответствовать агрегированной информации из полей ```wikipedia_summary_..```, ```heritage_status_..```, ```year_built```, ```architect_..```, ```architectural_style_..```

```python
example_data = [
  {
    "image_path": "/content/anowds3mbfkkwcosk4wcsssgo.jpg",
    "ground_truth": "Московский Кремль — древнейшая часть Москвы, охраняемый архитектурный ансамбль. Включает Успенский собор, где coronation царей происходило. Был резиденцией российских императоров и советских лидеров.",
    "name": "Московский Кремль"
  }
]
```
**Изображения счкачиваются в** ```download_images.py```

In [1]:
import json
import pandas as pd
import ast
import os
from tqdm import tqdm

In [5]:
LAND_PATH = "/Users/anastasiya/Documents/AITourGuide/setup_data_v3/data/landmarks_data_wiki.json"
TRAIN_PATH = "/Users/anastasiya/Documents/AITourGuide/data/train/train.csv"
IMAGES_DIR = "/Users/anastasiya/Documents/AITourGuide/images"

In [6]:
# загружаем json с отфильтрованными достопримечательностями и их описанием
# логика сбора данных: ...
with open(LAND_PATH, mode='r', encoding='utf-8') as f:
    data = json.load(f)

In [7]:
# пример отфильрованных данных
data[0]

{'wikidata_id': 'Q78',
 'name_en': 'Basel',
 'name_ru': 'Базель',
 'name_de': 'Basel',
 'wikidata_description_en': 'city on the Rhine in Switzerland',
 'wikidata_description_ru': 'город в Швейцарии, административный центр полукантона Базель-Штадт',
 'coordinates': {'latitude': 47.560555555556, 'longitude': 7.5905555555556},
 'country_ru': 'Швейцария',
 'country_en': 'Switzerland',
 'city_ru': 'Базель-Штадт',
 'city_en': 'Canton of Basel-Stadt',
 'heritage_status_ru': 'Объекты культурного наследия Швейцарии',
 'heritage_status_en': 'Swiss townscape worthy of protection',
 'website': 'https://www.bs.ch/',
 'wikipedia_url_en': 'https://en.wikipedia.org/wiki/Basel',
 'wikipedia_url_ru': 'https://ru.wikipedia.org/wiki/Базель',
 'wikipedia_url_de': 'https://de.wikipedia.org/wiki/Basel',
 'landmark_type': {'ru': 'достопримечательность',
  'en': 'landmark',
  'type_id': 'landmark'},
 'wikipedia_summary_en': "Basel ( BAH-zəl; German: [ˈbaːzl̩] ), also known as Basle (English:  BAHL), is a city 

In [5]:
# загружаем Google датасет со всеми достопримечательностями (id и ссылка на изображение)
train_df = pd.read_csv(TRAIN_PATH)

In [6]:
train_df.head()

,id,url,landmark_id
0,6e158a47eb2ca3f6,https://upload.wikimedia.org/wikipedia/commons...,142820
1,202cd79556f30760,http://upload.wikimedia.org/wikipedia/commons/...,104169
2,3ad87684c99c06e1,http://upload.wikimedia.org/wikipedia/commons/...,37914
3,e7f70e9c61e66af3,https://upload.wikimedia.org/wikipedia/commons...,102140
4,4072182eddd0100e,https://upload.wikimedia.org/wikipedia/commons...,2474


In [7]:
# лист с id достопримечательностей
collected_landmark_id = [landmark['landmark_id'] for landmark in data]
# датафрейм со ссылками на выбранные достопримечательности
df_collected_landmark_id_imgs = train_df[train_df.landmark_id.isin(collected_landmark_id)].copy()

In [8]:
df_collected_landmark_id_imgs = df_collected_landmark_id_imgs.sort_values(['landmark_id', 'url'])
df_collected_landmark_id_imgs['name'] = df_collected_landmark_id_imgs.groupby('landmark_id').cumcount() + 1
df_collected_landmark_id_imgs['name'] = df_collected_landmark_id_imgs['landmark_id'].astype(str) + '_' + df_collected_landmark_id_imgs['name'].astype(str) + '.jpg'

In [9]:
df_collected_landmark_id_imgs_gr = df_collected_landmark_id_imgs.groupby('landmark_id', as_index=False).agg({'url': list, 'name': list})

In [10]:
df_collected_landmark_id_imgs_gr.head()

,landmark_id,url,name
0,10,[http://upload.wikimedia.org/wikipedia/commons...,"[10_1.jpg, 10_2.jpg, 10_3.jpg, 10_4.jpg, 10_5...."
1,14,[https://upload.wikimedia.org/wikipedia/common...,"[14_1.jpg, 14_2.jpg]"
2,18,[http://upload.wikimedia.org/wikipedia/commons...,"[18_1.jpg, 18_2.jpg, 18_3.jpg, 18_4.jpg, 18_5...."
3,22,[http://upload.wikimedia.org/wikipedia/commons...,"[22_1.jpg, 22_2.jpg, 22_3.jpg, 22_4.jpg, 22_5...."
4,25,[http://upload.wikimedia.org/wikipedia/commons...,"[25_1.jpg, 25_2.jpg, 25_3.jpg, 25_4.jpg, 25_5...."


In [12]:
df_collected_landmark_id_imgs_gr.shape

(52267, 3)

In [ ]:
# Скачиваем изображения на основе df_collected_landmark_id_imgs_gr.csv
df_collected_landmark_id_imgs_gr.to_csv('data/df_collected_landmark_id_imgs_gr.csv', index=False)

# Клеим к данным image_path

Оставляем только примеры с изображениями и не пустым описанием

In [2]:
df_collected_landmark_id_imgs_gr = pd.read_csv('data/df_collected_landmark_id_imgs_gr.csv')
df_collected_landmark_id_imgs_gr.head()

,landmark_id,url,name
0,10,['http://upload.wikimedia.org/wikipedia/common...,"['10_1.jpg', '10_2.jpg', '10_3.jpg', '10_4.jpg..."
1,14,['https://upload.wikimedia.org/wikipedia/commo...,"['14_1.jpg', '14_2.jpg']"
2,18,['http://upload.wikimedia.org/wikipedia/common...,"['18_1.jpg', '18_2.jpg', '18_3.jpg', '18_4.jpg..."
3,22,['http://upload.wikimedia.org/wikipedia/common...,"['22_1.jpg', '22_2.jpg', '22_3.jpg', '22_4.jpg..."
4,25,['http://upload.wikimedia.org/wikipedia/common...,"['25_1.jpg', '25_2.jpg', '25_3.jpg', '25_4.jpg..."


In [8]:
data_with_img = []
for landmark in tqdm(data):
    image_names = df_collected_landmark_id_imgs_gr[df_collected_landmark_id_imgs_gr['landmark_id'] == landmark['landmark_id']]['name'].values[0]
    image_names = ast.literal_eval(image_names)
    
    # Фильтрация существующих изображений
    existing_images = [img for img in image_names 
                      if os.path.exists(os.path.join(IMAGES_DIR, img))]
    
    if len(existing_images) > 0 \
    and landmark.get('wikipedia_summary_en', None) is not None \
        and landmark.get('name_ru', landmark.get('name_en')) is not None:
        landmark['image_path'] = existing_images
        data_with_img.append(landmark)


100%|██████████| 52267/52267 [01:14<00:00, 705.55it/s]


In [9]:
data_with_img[0]

{'wikidata_id': 'Q78',
 'name_en': 'Basel',
 'name_ru': 'Базель',
 'name_de': 'Basel',
 'wikidata_description_en': 'city on the Rhine in Switzerland',
 'wikidata_description_ru': 'город в Швейцарии, административный центр полукантона Базель-Штадт',
 'coordinates': {'latitude': 47.560555555556, 'longitude': 7.5905555555556},
 'country_ru': 'Швейцария',
 'country_en': 'Switzerland',
 'city_ru': 'Базель-Штадт',
 'city_en': 'Canton of Basel-Stadt',
 'heritage_status_ru': 'Объекты культурного наследия Швейцарии',
 'heritage_status_en': 'Swiss townscape worthy of protection',
 'website': 'https://www.bs.ch/',
 'wikipedia_url_en': 'https://en.wikipedia.org/wiki/Basel',
 'wikipedia_url_ru': 'https://ru.wikipedia.org/wiki/Базель',
 'wikipedia_url_de': 'https://de.wikipedia.org/wiki/Basel',
 'landmark_type': {'ru': 'достопримечательность',
  'en': 'landmark',
  'type_id': 'landmark'},
 'wikipedia_summary_en': "Basel ( BAH-zəl; German: [ˈbaːzl̩] ), also known as Basle (English:  BAHL), is a city 

In [10]:
with open('data/landmarks_data_wiki_with_img.json', mode='w', encoding='utf-8') as f:
    json.dump(data_with_img, f, ensure_ascii=False, indent=2)

In [12]:
print(f"Количество собранных достопримечательностей с фото {len(data_with_img)}")
print(f"Доля собранных достопримечательностей с фото {round(len(data_with_img)/len(data)*100, 2)}%")

Количество собранных достопримечательностей с фото 33503
Доля собранных достопримечательностей с фото 64.1%


Далее landmarks_data_wiki_with_img.json отправляем на фильтрацию описанийи фото